In [9]:
import cv2
import dlib
import numpy as np
import os
import sys

# Cargar modelos (deben estar en el mismo directorio que el notebook)
predictor_path = 'shape_predictor_68_face_landmarks.dat'
face_rec_model_path = 'dlib_face_recognition_resnet_model_v1.dat'

detector = dlib.get_frontal_face_detector()
sp = dlib.shape_predictor(predictor_path)
facerec = dlib.face_recognition_model_v1(face_rec_model_path)

def cargar_rostros_y_embeddings(carpeta_rostros='rostros'):
    carpeta_absoluta = os.path.join(os.getcwd(), carpeta_rostros)
    print(f"Cargando imágenes desde: {carpeta_absoluta}")

    if not os.path.exists(carpeta_absoluta):
        print(f"Error: No existe la carpeta {carpeta_absoluta}")
        sys.exit(1)

    nombres = []
    descriptores = []

    for archivo in os.listdir(carpeta_absoluta):
        if archivo.lower().endswith(('.jpg', '.png')):
            img_path = os.path.join(carpeta_absoluta, archivo)
            print(f"Cargando imagen: {img_path}")
            img = dlib.load_rgb_image(img_path)

            dets = detector(img, 1)
            if len(dets) > 0:
                shape = sp(img, dets[0])
                face_descriptor = facerec.compute_face_descriptor(img, shape)
                descriptores.append(np.array(face_descriptor))
                nombre = os.path.splitext(archivo)[0]
                nombres.append(nombre)
            else:
                print(f"No se detectó rostro en {archivo}")

    return nombres, descriptores

nombres_registrados, descriptores_registrados = cargar_rostros_y_embeddings()

def reconocer_rostro(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    dets = detector(rgb, 1)

    for d in dets:
        shape = sp(rgb, d)
        face_descriptor = facerec.compute_face_descriptor(rgb, shape)
        descriptor_actual = np.array(face_descriptor)

        distancias = [np.linalg.norm(descriptor_actual - desc) for desc in descriptores_registrados]
        idx = np.argmin(distancias)

        if distancias[idx] < 0.6:
            nombre = nombres_registrados[idx]
        else:
            nombre = "Desconocido"

        x1, y1, x2, y2 = d.left(), d.top(), d.right(), d.bottom()
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, nombre, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    return frame

# Captura cámara (puede no funcionar en todos los notebooks)
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("No se pudo abrir la cámara.")
else:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_procesado = reconocer_rostro(frame)
        cv2.imshow("Reconocimiento Facial", frame_procesado)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


Cargando imágenes desde: d:\Documentos upao\7mo Ciclo\Percepcion Computacional\proyecto_asistencia\rostros
Cargando imagen: d:\Documentos upao\7mo Ciclo\Percepcion Computacional\proyecto_asistencia\rostros\Adrian Nicolas.jpg
Cargando imagen: d:\Documentos upao\7mo Ciclo\Percepcion Computacional\proyecto_asistencia\rostros\AlejandroOliden.jpg
Cargando imagen: d:\Documentos upao\7mo Ciclo\Percepcion Computacional\proyecto_asistencia\rostros\AngelOncoy.jpg
Cargando imagen: d:\Documentos upao\7mo Ciclo\Percepcion Computacional\proyecto_asistencia\rostros\AnthonyLezcano.jpg
Cargando imagen: d:\Documentos upao\7mo Ciclo\Percepcion Computacional\proyecto_asistencia\rostros\BensonHilario.jpg
Cargando imagen: d:\Documentos upao\7mo Ciclo\Percepcion Computacional\proyecto_asistencia\rostros\BryanMarin.jpg
Cargando imagen: d:\Documentos upao\7mo Ciclo\Percepcion Computacional\proyecto_asistencia\rostros\CarlosMezones.jpg
Cargando imagen: d:\Documentos upao\7mo Ciclo\Percepcion Computacional\proye